In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import torch.nn as nn
import torchvision.models as models


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
transform = transforms.Compose([
    transforms.Resize((150, 150)),        # Resize images to a fixed size
    transforms.ToTensor(),                # Convert image to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize
])

In [ ]:
# Load the dataset
dataset_directory = "png_modified"  # Path to your image dataset
dataset = datasets.ImageFolder(root=dataset_directory, transform=transform)


In [ ]:
from torch.utils.data import random_split

# Split dataset into training and validation (80/20 split)
train_size = int(0.8 * len(dataset))  # 80% for training
val_size = len(dataset) - train_size  # 20% for validation

train_data, val_data = random_split(dataset, [train_size, val_size])




In [ ]:
# Print class labels to verify they're in alphabetical order
print(f"Class labels (alphabetically sorted): {dataset.classes}")


In [ ]:
# Create DataLoader objects for both training and validation sets
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
# Initialize the model (pre-trained ResNet18)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)  # Use pre-trained weights

# Modify the final fully connected layer to match the number of classes in your dataset
model.fc = nn.Linear(model.fc.in_features, len(dataset.classes))

# Move the model to the GPU (if available)
model = model.to(device)


# Unfreeze all layers (i.e., make them trainable)
for param in model.parameters():
    param.requires_grad = True


In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Learning rate can be adjusted
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, verbose=True)


In [ ]:
# Training loop
num_epochs = 50

for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader)}")

    # Validation phase
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Validation Accuracy: {accuracy:.2f}%")

    # Step the scheduler based on validation loss (optional)
    scheduler.step(1 - accuracy)

In [ ]:
torch.save(model.state_dict(), 'resnet18_trained_modelv3.pth')